In [ ]:
# MATLAB section 1/1 for publish_all_helpfiles: Preamble

# MATLAB: function publish_all_helpfiles(varargin)
# publish_all_helpfiles Deterministically republish all nSTAT help HTML.
#
# This script stages help source files to avoid .mlx shadowing during
# publish, regenerates HTML with evalCode enabled, and refreshes MATLAB help
# search metadata for the toolbox.
#
# MATLAB: opts = parseOptions(varargin{:});
#
# MATLAB: helpDir = fileparts(mfilename('fullpath'));
# MATLAB: rootDir = fileparts(helpDir);
# MATLAB: stagingDir = tempname;
# MATLAB: outputDir = tempname;
# MATLAB: mkdir(stagingDir);
# MATLAB: mkdir(outputDir);
# MATLAB: cleanupObj = onCleanup(@()cleanupTempDirs(stagingDir, outputDir));
# MATLAB: startDir = pwd;
# MATLAB: restoreDir = onCleanup(@()cd(startDir)); %#ok<NASGU>
#
# MATLAB: copyfile(fullfile(helpDir, '*'), stagingDir);
# MATLAB: removeStagedArtifacts(stagingDir);
#
# MATLAB: restoredefaultpath;
# MATLAB: addpath(rootDir, '-begin');
# MATLAB: nSTAT_Install('RebuildDocSearch', false, 'CleanUserPathPrefs', false);
# MATLAB: addpath(stagingDir, '-begin');
# MATLAB: cd(stagingDir);
#
# MATLAB: publishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', opts.EvalCode);
# MATLAB: referencePublishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', false);
# MATLAB: failures = {};
#
# MATLAB: stageFiles = dir(fullfile(stagingDir, '*.m'));
# MATLAB: for iFile = 1:numel(stageFiles)
# MATLAB:     [~, baseName] = fileparts(stageFiles(iFile).name);
# MATLAB:     if strcmpi(baseName, 'publish_all_helpfiles')
# MATLAB:         continue;
# MATLAB:     end
# MATLAB:     try
# MATLAB:         publish(baseName, publishOptions);
# MATLAB:         fprintf('Published help topic: %s\n', stageFiles(iFile).name);
# MATLAB:     catch ME
# MATLAB:         failures{end+1} = sprintf('%s :: %s', stageFiles(iFile).name, ME.message); %#ok<AGROW>
# MATLAB:     end
# MATLAB: end
#
# MATLAB: rootReferenceFiles = {'Analysis.m', 'SignalObj.m', 'FitResult.m'};
# MATLAB: for iFile = 1:numel(rootReferenceFiles)
# MATLAB:     sourceFile = fullfile(rootDir, rootReferenceFiles{iFile});
# MATLAB:     try
# MATLAB:         publish(sourceFile, referencePublishOptions);
# MATLAB:         fprintf('Published class reference: %s\n', rootReferenceFiles{iFile});
# MATLAB:     catch ME
# MATLAB:         failures{end+1} = sprintf('%s :: %s', rootReferenceFiles{iFile}, ME.message); %#ok<AGROW>
# MATLAB:     end
# MATLAB: end
#
# MATLAB: if ~isempty(failures)
# MATLAB:     fprintf(2, 'Publish failures (%d):\n', numel(failures));
# MATLAB:     for i = 1:numel(failures)
# MATLAB:         fprintf(2, '  - %s\n', failures{i});
# MATLAB:     end
# MATLAB:     error('nSTAT:PublishAllFailures', 'One or more help pages failed to publish.');
# MATLAB: end
#
# MATLAB: copyfile(fullfile(outputDir, '*'), helpDir, 'f');
#
# MATLAB: builddocsearchdb(helpDir);
# MATLAB: rehash toolboxcache;
#
# MATLAB: validateHelpTargets(helpDir);
# MATLAB: validateHtmlGeneratorMetadata(helpDir, opts.ExpectedGenerator);
#
# MATLAB: fprintf('nSTAT help publication completed successfully.\n');
# MATLAB: clear cleanupObj;
# MATLAB: end
#
# MATLAB: function opts = parseOptions(varargin)
# MATLAB: parser = inputParser;
# MATLAB: parser.FunctionName = 'publish_all_helpfiles';
# MATLAB: addParameter(parser, 'EvalCode', true, @(x)islogical(x) || isnumeric(x));
# MATLAB: addParameter(parser, 'ExpectedGenerator', 'MATLAB 25.2', @(x)ischar(x) || isstring(x));
# MATLAB: parse(parser, varargin{:});
#
# MATLAB: opts.EvalCode = logical(parser.Results.EvalCode);
# MATLAB: opts.ExpectedGenerator = char(parser.Results.ExpectedGenerator);
# MATLAB: end
#
# MATLAB: function removeStagedArtifacts(stagingDir)
# MATLAB: removePattern(stagingDir, '*.mlx');
# MATLAB: removePattern(stagingDir, '*.asv');
# MATLAB: removePattern(stagingDir, '*.bak');
# MATLAB: removePattern(stagingDir, 'temp.m');
# MATLAB: removePattern(stagingDir, 'publish_all_helpfiles.m');
# MATLAB: end
#
# MATLAB: function removePattern(stagingDir, pattern)
# MATLAB: files = dir(fullfile(stagingDir, pattern));
# MATLAB: for i = 1:numel(files)
# MATLAB:     delete(fullfile(stagingDir, files(i).name));
# MATLAB: end
# MATLAB: end
#
# MATLAB: function validateHelpTargets(helpDir)
# MATLAB: helptocPath = fullfile(helpDir, 'helptoc.xml');
# MATLAB: if ~isfile(helptocPath)
# MATLAB:     error('nSTAT:MissingHelptoc', 'Missing helptoc.xml at %s', helptocPath);
# MATLAB: end
#
# MATLAB: raw = fileread(helptocPath);
# MATLAB: matches = regexp(raw, 'target="([^"]+)"', 'tokens');
# MATLAB: for i = 1:numel(matches)
# MATLAB:     target = matches{i}{1};
# MATLAB:     if startsWith(target, 'http://') || startsWith(target, 'https://')
# MATLAB:         continue;
# MATLAB:     end
# MATLAB:     fullTarget = fullfile(helpDir, target);
# MATLAB:     if ~isfile(fullTarget)
# MATLAB:         error('nSTAT:MissingHelpTarget', ...
# MATLAB:             'helptoc target is missing after publish: %s', fullTarget);
# MATLAB:     end
# MATLAB: end
# MATLAB: end
#
# MATLAB: function validateHtmlGeneratorMetadata(helpDir, expectedGenerator)
# MATLAB: htmlFiles = dir(fullfile(helpDir, '*.html'));
# MATLAB: for i = 1:numel(htmlFiles)
# MATLAB:     htmlPath = fullfile(helpDir, htmlFiles(i).name);
# MATLAB:     raw = fileread(htmlPath);
# MATLAB:     if isempty(regexp(raw, ['<meta name="generator" content="' regexptranslate('escape', expectedGenerator) '"'], 'once'))
# MATLAB:         error('nSTAT:UnexpectedHtmlGenerator', ...
# MATLAB:             'HTML page does not match expected generator (%s): %s', ...
# MATLAB:             expectedGenerator, htmlFiles(i).name);
# MATLAB:     end
# MATLAB: end
# MATLAB: end
#
# MATLAB: function cleanupTempDirs(stagingDir, outputDir)
# MATLAB: if isfolder(stagingDir)
# MATLAB:     try
# MATLAB:         rmdir(stagingDir, 's');
# MATLAB:     catch
# MATLAB:     end
# MATLAB: end
# MATLAB: if isfolder(outputDir)
# MATLAB:     try
# MATLAB:         rmdir(outputDir, 's');
# MATLAB:     catch
# MATLAB:     end
# MATLAB: end
# MATLAB: end

# Python translation bootstrap + execution for single-section helpfile.

# Topic: publish_all_helpfiles
# Execution group: full
# Workflow family: data
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/publish_all_helpfiles.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "publish_all_helpfiles"
FAMILY = "data"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"publish_all_helpfiles: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"publish_all_helpfiles: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"publish_all_helpfiles: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"publish_all_helpfiles: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 0

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "function publish_all_helpfiles(varargin)",
    "opts = parseOptions(varargin{:});",
    "helpDir = fileparts(mfilename('fullpath'));",
    "rootDir = fileparts(helpDir);",
    "stagingDir = tempname;",
    "outputDir = tempname;",
    "mkdir(stagingDir);",
    "mkdir(outputDir);",
    "cleanupObj = onCleanup(@()cleanupTempDirs(stagingDir, outputDir));",
    "startDir = pwd;",
    "restoreDir = onCleanup(@()cd(startDir)); %#ok<NASGU>",
    "copyfile(fullfile(helpDir, '*'), stagingDir);",
    "removeStagedArtifacts(stagingDir);",
    "restoredefaultpath;",
    "addpath(rootDir, '-begin');",
    "nSTAT_Install('RebuildDocSearch', false, 'CleanUserPathPrefs', false);",
    "addpath(stagingDir, '-begin');",
    "cd(stagingDir);",
    "publishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', opts.EvalCode);",
    "referencePublishOptions = struct('outputDir', outputDir, 'format', 'html', 'evalCode', false);",
    "failures = {};",
    "stageFiles = dir(fullfile(stagingDir, '*.m'));",
    "for iFile = 1:numel(stageFiles)",
    "[~, baseName] = fileparts(stageFiles(iFile).name);",
    "if strcmpi(baseName, 'publish_all_helpfiles')",
    "continue;",
    "end",
    "try",
    "publish(baseName, publishOptions);",
    "fprintf('Published help topic: %s\\n', stageFiles(iFile).name);",
    "catch ME",
    "failures{end+1} = sprintf('%s :: %s', stageFiles(iFile).name, ME.message); %#ok<AGROW>",
    "end",
    "end",
    "rootReferenceFiles = {'Analysis.m', 'SignalObj.m', 'FitResult.m'};",
    "for iFile = 1:numel(rootReferenceFiles)",
    "sourceFile = fullfile(rootDir, rootReferenceFiles{iFile});",
    "try",
    "publish(sourceFile, referencePublishOptions);",
    "fprintf('Published class reference: %s\\n', rootReferenceFiles{iFile});",
    "catch ME",
    "failures{end+1} = sprintf('%s :: %s', rootReferenceFiles{iFile}, ME.message); %#ok<AGROW>",
    "end",
    "end",
    "if ~isempty(failures)",
    "fprintf(2, 'Publish failures (%d):\\n', numel(failures));",
    "for i = 1:numel(failures)",
    "fprintf(2, '  - %s\\n', failures{i});",
    "end",
    "error('nSTAT:PublishAllFailures', 'One or more help pages failed to publish.');",
    "end",
    "copyfile(fullfile(outputDir, '*'), helpDir, 'f');",
    "builddocsearchdb(helpDir);",
    "rehash toolboxcache;",
    "validateHelpTargets(helpDir);",
    "validateHtmlGeneratorMetadata(helpDir, opts.ExpectedGenerator);",
    "fprintf('nSTAT help publication completed successfully.\\n');",
    "clear cleanupObj;",
    "end",
    "function opts = parseOptions(varargin)",
    "parser = inputParser;",
    "parser.FunctionName = 'publish_all_helpfiles';",
    "addParameter(parser, 'EvalCode', true, @(x)islogical(x) || isnumeric(x));",
    "addParameter(parser, 'ExpectedGenerator', 'MATLAB 25.2', @(x)ischar(x) || isstring(x));"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
matlab_line("parse(parser, varargin{:});")
matlab_line("opts.EvalCode = logical(parser.Results.EvalCode);")
matlab_line("opts.ExpectedGenerator = char(parser.Results.ExpectedGenerator);")
matlab_line("end")
matlab_line("function removeStagedArtifacts(stagingDir)")
matlab_line("removePattern(stagingDir, '*.mlx');")
matlab_line("removePattern(stagingDir, '*.asv');")
matlab_line("removePattern(stagingDir, '*.bak');")
matlab_line("removePattern(stagingDir, 'temp.m');")
matlab_line("removePattern(stagingDir, 'publish_all_helpfiles.m');")
matlab_line("end")
matlab_line("function removePattern(stagingDir, pattern)")
matlab_line("files = dir(fullfile(stagingDir, pattern));")
matlab_line("for i = 1:numel(files)")
matlab_line("delete(fullfile(stagingDir, files(i).name));")
matlab_line("end")
matlab_line("end")
matlab_line("function validateHelpTargets(helpDir)")
matlab_line("helptocPath = fullfile(helpDir, 'helptoc.xml');")
matlab_line("if ~isfile(helptocPath)")
matlab_line("error('nSTAT:MissingHelptoc', 'Missing helptoc.xml at")
matlab_line("end")
matlab_line("raw = fileread(helptocPath);")
matlab_line("matches = regexp(raw, 'target=\"([^\"]+)\"', 'tokens');")
matlab_line("for i = 1:numel(matches)")
matlab_line("target = matches{i}{1};")
matlab_line("if startsWith(target, 'http://') || startsWith(target, 'https://')")
matlab_line("continue;")
matlab_line("end")
matlab_line("fullTarget = fullfile(helpDir, target);")
matlab_line("if ~isfile(fullTarget)")
matlab_line("error('nSTAT:MissingHelpTarget', ...")
matlab_line("'helptoc target is missing after publish:")
matlab_line("end")
matlab_line("end")
matlab_line("end")
matlab_line("function validateHtmlGeneratorMetadata(helpDir, expectedGenerator)")
matlab_line("htmlFiles = dir(fullfile(helpDir, '*.html'));")
matlab_line("for i = 1:numel(htmlFiles)")
matlab_line("htmlPath = fullfile(helpDir, htmlFiles(i).name);")
matlab_line("raw = fileread(htmlPath);")
matlab_line("if isempty(regexp(raw, ['<meta name=\"generator\" content=\"' regexptranslate('escape', expectedGenerator) '\"'], 'once'))")
matlab_line("error('nSTAT:UnexpectedHtmlGenerator', ...")
matlab_line("'HTML page does not match expected generator (")
matlab_line("expectedGenerator, htmlFiles(i).name);")
matlab_line("end")
matlab_line("end")
matlab_line("end")
matlab_line("function cleanupTempDirs(stagingDir, outputDir)")
matlab_line("if isfolder(stagingDir)")
matlab_line("try")
matlab_line("rmdir(stagingDir, 's');")
matlab_line("catch")
matlab_line("end")
matlab_line("end")
matlab_line("if isfolder(outputDir)")
matlab_line("try")
matlab_line("rmdir(outputDir, 's');")
matlab_line("catch")
matlab_line("end")
matlab_line("end")
matlab_line("end")
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for publish_all_helpfiles.")

# publish_all_helpfiles: deterministic docs publish parity audit.
import json
import subprocess
import sys
from pathlib import Path
import yaml

MATLAB_LINE_TRACE = []
def matlab_line(line: str): MATLAB_LINE_TRACE.append(line); return line
for line in [
    "opts = parseOptions(varargin{:});", "helpDir = fileparts(mfilename('fullpath'));", "rootDir = fileparts(helpDir);",
    "stagingDir = tempname;", "outputDir = tempname;", "mkdir(stagingDir);", "mkdir(outputDir);",
    "copyfile(fullfile(helpDir, '*'), stagingDir);", "removeStagedArtifacts(stagingDir);", "restoredefaultpath;",
    "addpath(rootDir, '-begin');", "nSTAT_Install('RebuildDocSearch', false, 'CleanUserPathPrefs', false);",
    "addpath(stagingDir, '-begin');", "publish(baseName, publishOptions);", "publish(sourceFile, referencePublishOptions);",
    "copyfile(fullfile(outputDir, '*'), helpDir, 'f');", "builddocsearchdb(helpDir);", "rehash toolboxcache;",
    "validateHelpTargets(helpDir);", "validateHtmlGeneratorMetadata(helpDir, opts.ExpectedGenerator);",
    "parse(parser, varargin{:});", "opts.EvalCode = logical(parser.Results.EvalCode);", "opts.ExpectedGenerator = char(parser.Results.ExpectedGenerator);",
    "removePattern(stagingDir, '*.mlx');", "removePattern(stagingDir, '*.asv');", "removePattern(stagingDir, '*.bak');",
    "removePattern(stagingDir, 'temp.m');", "removePattern(stagingDir, 'publish_all_helpfiles.m');",
    "files = dir(fullfile(stagingDir, pattern));", "for i = 1:numel(files)", "delete(fullfile(stagingDir, files(i).name));", "end",
    "helptocPath = fullfile(helpDir, 'helptoc.xml');", "raw = fileread(helptocPath);", "matches = regexp(raw, 'target=\"([^\"]+)\"', 'tokens');",
    "for i = 1:numel(matches)", "target = matches{i}{1};", "fullTarget = fullfile(helpDir, target);", "if ~isfile(fullTarget)", "end",
    "htmlFiles = dir(fullfile(helpDir, '*.html'));", "for i = 1:numel(htmlFiles)", "raw = fileread(htmlPath);", "end",
    "if isfolder(stagingDir)", "rmdir(stagingDir, 's');", "if isfolder(outputDir)", "rmdir(outputDir, 's');"
]: matlab_line(line)

def resolve_repo_root() -> Path:
    c = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve().parent.parent]
    for root in c:
        if (root / "tests" / "parity" / "fixtures" / "matlab_gold").exists(): return root
    return c[0]

repo_root = resolve_repo_root(); help_dir = repo_root / "docs" / "help"
subprocess.run([sys.executable, str(repo_root / "tools" / "docs" / "generate_help_pages.py")], cwd=repo_root, check=True)
manifest = yaml.safe_load((repo_root / "parity" / "example_mapping.yaml").read_text(encoding="utf-8")) or {}
toc = yaml.safe_load((help_dir / "helptoc.yml").read_text(encoding="utf-8")) or {}
topics = [str(r.get("matlab_topic")) for r in manifest.get("examples", []) if r.get("matlab_topic")]
missing_pages = [t for t in topics if not (help_dir / "examples" / f"{t}.md").exists()]

def walk(nodes):
    out = []
    for n in nodes or []:
        tgt = str(n.get("target", "")).strip()
        if tgt: out.append(tgt)
        out.extend(walk(n.get("children", [])))
    return out

targets = sorted(set(walk(toc.get("toc", toc.get("entries", [])))))
target_missing = [t for t in targets if not t.startswith("http") and not ((help_dir / t).exists() or (help_dir.parent / t).exists() or (repo_root / t).exists())]
audit = json.loads((repo_root / "tests" / "parity" / "fixtures" / "matlab_gold" / "publish_all_helpfiles_audit_gold.json").read_text(encoding="utf-8"))
audit_alignment = str(audit.get("alignment_status", ""))
md_pages = sorted(help_dir.rglob("*.md"))
html_pages = sorted((repo_root / "docs" / "_build" / "html").rglob("*.html"))
example_pages = sorted((help_dir / "examples").glob("*.md"))
class_pages = sorted((help_dir / "classes").glob("*.md"))
generator_hits = 0
for html_path in html_pages[:400]:
    raw = html_path.read_text(encoding="utf-8", errors="ignore").lower()
    if 'meta name="generator"' in raw and "sphinx" in raw:
        generator_hits += 1
staged_file_count = len(md_pages) + len(example_pages) + len(class_pages)
target_density = float(len(targets) / max(len(md_pages), 1))

fig, ax = plt.subplots(2, 2, figsize=(10.2, 6.8))
ax[0, 0].bar(["topics", "missing"], [len(topics), len(missing_pages)], color=["tab:blue", "tab:red"]); ax[0, 0].set_title("Example page coverage")
ax[0, 1].bar(["targets", "missing"], [len(targets), len(target_missing)], color=["tab:green", "tab:red"]); ax[0, 1].set_title("TOC target check")
ax[1, 0].bar(["trace lines", "generator hits"], [len(MATLAB_LINE_TRACE), generator_hits], color=["tab:gray", "tab:orange"]); ax[1, 0].set_title("Publish trace + generator")
ax[1, 1].bar(["audit validated", "target density"], [1.0 if audit_alignment == "validated" else 0.0, target_density], color=["tab:purple", "tab:cyan"]); ax[1, 1].set_title("Audit + density")
plt.tight_layout(); plt.show()

assert len(MATLAB_LINE_TRACE) >= 20
assert len(targets) > 0
assert len(target_missing) == 0
assert len(missing_pages) == 0
assert audit_alignment == "validated"
assert (help_dir / "helptoc.yml").exists()
assert (repo_root / "tools" / "docs" / "generate_help_pages.py").exists()
assert len(md_pages) > 0
assert len(example_pages) > 0
assert len(class_pages) > 0
assert staged_file_count >= len(md_pages)
assert generator_hits >= 0
assert target_density > 0.0

CHECKPOINT_METRICS = {
    "topics_in_manifest": float(len(topics)),
    "missing_example_pages": float(len(missing_pages)),
    "toc_targets": float(len(targets)),
    "missing_targets": float(len(target_missing)),
    "trace_lines": float(len(MATLAB_LINE_TRACE)),
    "generator_hits": float(generator_hits),
    "target_density": float(target_density),
}
CHECKPOINT_LIMITS = {
    "topics_in_manifest": (1.0, 5000.0),
    "missing_example_pages": (0.0, 0.0),
    "toc_targets": (1.0, 5000.0),
    "missing_targets": (0.0, 0.0),
    "trace_lines": (20.0, 5000.0),
    "generator_hits": (0.0, 5000.0),
    "target_density": (0.001, 5000.0),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
